# 08. Exposure and expected annual exposure

This notebook overlays the 20 classified hazard maps on buildings, roads and critical facilities and derives the exposure-frequency relationships, the frequency amplification factor and the expected annual exposure (EAE) at city and ward scale. It produces the exposure tables, the exposure-frequency curves, the land-use heat map, the construction-type table and the risk tables.


## Configuration

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import matplotlib.pyplot as plt
%matplotlib inline

warnings.filterwarnings("ignore")
pd.set_option("display.width", 190)
TRAPZ = getattr(np, "trapezoid", getattr(np, "trapz", None))   # NumPy 2 renamed trapz to trapezoid

ROOT = Path("..")
RETURN_PERIODS = ["2yr", "10yr", "50yr", "100yr"]
RP_YEARS = {"2yr": 2, "5yr": 5, "10yr": 10, "25yr": 25, "50yr": 50, "100yr": 100}
SCENARIOS = ["Observed", "SSP245_2021_2050", "SSP245_2051_2080", "SSP585_2021_2050", "SSP585_2051_2080"]
BASELINE = "Observed"
FUTURE_CASE = "SSP585_2051_2080"       # case used for the land-use table and the changes in EAE

CLASSIFIED_PATTERN = str(ROOT / "results" / "07_ufhi" / "UFHI_classified_{rp}_{scen}.tif")
BUILDINGS = ROOT / "data" / "buildings" / "buildings_study_wards.shp"
USAGE_FIELD = "Struse1T"; SUBUSE_FIELD = "Struse2T"; STRUCT_FIELD = "STRUCTYPE"; WARD_FIELD = "WARD_NO"
ROADS = {name: ROOT / "data" / "roads" / f"{name}.shp"
         for name in ["Pacca", "Kacharoad", "Island", "Footpath", "Semipacca"]}

# critical facilities: usage category in CRIT_USAGE1 or usage sub-category in CRIT_USAGE2
CRIT_USAGE1 = ["Education & Research", "Community Service", "Governmental Services"]
CRIT_USAGE2 = ["School", "College", "Madrasa", "Health Service", "Hospital", "Protective Service", "Religious"]
NON_DURABLE_TYPES = ["Katcha", "Semi Pucca"]

HIGH = [4, 5]                          # High and Very High classes
OUT_DIR = ROOT / "results" / "08_exposure"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASES = [(rp, s) for rp in RETURN_PERIODS for s in SCENARIOS]
def cpath(rp, s): return CLASSIFIED_PATTERN.format(rp=rp, scen=s)

missing = [cpath(rp, s) for rp, s in CASES if not Path(cpath(rp, s)).exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} classified raster(s) not found, for example {missing[0]}")
print(f"{len(CASES)} cases = {len(RETURN_PERIODS)} return periods x {len(SCENARIOS)} climates")

## 1. Asset inventories

In [ ]:
with rasterio.open(cpath(*CASES[0])) as s:
    CRS = s.crs; PX = abs(s.transform.a * s.transform.e)        # cell area (m2)

bld = gpd.read_file(BUILDINGS).to_crs(CRS)
bld["is_crit"] = bld[USAGE_FIELD].isin(CRIT_USAGE1) | bld[SUBUSE_FIELD].isin(CRIT_USAGE2)
roads = gpd.GeoDataFrame(
    pd.concat([gpd.read_file(p).to_crs(CRS)[["geometry"]].assign(_type=t) for t, p in ROADS.items()],
              ignore_index=True), crs=CRS)
print("buildings", len(bld), "| critical facilities", int(bld["is_crit"].sum()), "| road features", len(roads))

## 2. Functions

- **Majority class:** each building is assigned the majority hazard class of the raster cells whose centres fall within its footprint.
- **Road area by class:** the number of classified cells within the road polygons multiplied by the cell area.
- **Expected annual exposure:** with $p_i = 1/T_i$ for the simulated return periods and an anchor of zero high-hazard exposure at $p = 1$, the integral of exposure over the annual exceedance probability is evaluated by the trapezoidal rule (Olsen et al. 2015),
  $$\mathrm{EAE} = \int_{0.01}^{1} E(p)\,dp \approx \sum_{i=1}^{N-1}\frac{E(p_i) + E(p_{i+1})}{2}\left(p_i - p_{i+1}\right)$$
  The integral is truncated at $p = 0.01$, so events rarer than the 100-year storm are not counted.
- **Equivalent return period:** the return period $T_{eq}$ at which a climate case reaches a target exposure, found by linear interpolation of $\ln T$ against exposure between the simulated return periods. The frequency amplification factor is $A_F = 100/T_{eq}$.

In [ ]:
def majority(gdf, path):
    zs = zonal_stats(gdf, path, stats=["majority"], nodata=0)
    return [int(z["majority"]) if z["majority"] is not None else np.nan for z in zs]

def road_area_by_class(path):
    zs = zonal_stats(roads, path, categorical=True, nodata=0); rec = []
    for i, z in enumerate(zs):
        for cls, cnt in (z or {}).items():
            rec.append({"type": roads.iloc[i]["_type"], "Class": int(cls), "ha": cnt * PX / 1e4})
    return pd.DataFrame(rec)

def EAE(value_by_rp):
    """Expected annual exposure: integral of exposure over the annual exceedance probability (1/T)."""
    T = np.array([RP_YEARS[r] for r in RETURN_PERIODS], float); p = 1.0 / T
    E = np.array([value_by_rp[r] for r in RETURN_PERIODS], float)
    p = np.append(p, 1.0); E = np.append(E, 0.0)          # anchor: zero high-hazard exposure at p = 1
    o = np.argsort(p); return float(TRAPZ(E[o], p[o]))

def EAE_intervals(value_by_rp):
    """Contribution of each probability interval to the EAE, from the anchor at p = 1 to the rarest event."""
    T = np.array([RP_YEARS[r] for r in RETURN_PERIODS], float)
    p = np.append(1.0, 1.0 / T); E = np.append(0.0, [value_by_rp[r] for r in RETURN_PERIODS])
    o = np.argsort(-p); p, E = p[o], E[o]
    return (E[:-1] + E[1:]) / 2 * (p[:-1] - p[1:])

def rp_shift(value_by_rp, target):
    """Return period at which the exposure equals 'target', and the amplification factor 100 / T_eq."""
    T = np.array([RP_YEARS[r] for r in RETURN_PERIODS], float); E = np.array([value_by_rp[r] for r in RETURN_PERIODS], float)
    o = np.argsort(E); Tf = float(np.exp(np.interp(target, E[o], np.log(T[o]))))
    return Tf, RP_YEARS["100yr"] / Tf

def high_pct(sub, rp, s):
    """Percentage of features in the high-hazard classes, among features with an assigned class."""
    col = f"{rp}__{s}"; v = sub[col].dropna()
    return round(100 * v.isin(HIGH).sum() / len(v), 2) if len(v) else np.nan

def high_cnt(sub, rp, s):
    return int(sub[f"{rp}__{s}"].isin(HIGH).sum())

## 3. Hazard class of each building

Buildings whose footprints lie on cells without a hazard value receive no class and are excluded from the percentage denominators.

In [ ]:
for rp, s in CASES:
    bld[f"{rp}__{s}"] = majority(bld, cpath(rp, s))
print(f"Buildings without a hazard class, 100-year {BASELINE}:", int(bld[f"100yr__{BASELINE}"].isna().sum()))

## 4. Building exposure

In [ ]:
bpct = pd.DataFrame({s: [high_pct(bld, rp, s) for rp in RETURN_PERIODS] for s in SCENARIOS}, index=RETURN_PERIODS)
bcnt = pd.DataFrame({s: [high_cnt(bld, rp, s) for rp in RETURN_PERIODS] for s in SCENARIOS}, index=RETURN_PERIODS)
bpct_inventory = (100 * bcnt / len(bld)).round(2)
print("Buildings in the High and Very High classes (% of buildings with an assigned class):"); display(bpct)
print("Buildings in the High and Very High classes (number):"); display(bcnt)
print(f"Buildings in the High and Very High classes (% of the full inventory, n = {len(bld)}):"); display(bpct_inventory)

## 5. Critical-facility exposure

In [ ]:
crit = bld[bld["is_crit"]]
cpct = pd.DataFrame({s: [high_pct(crit, rp, s) for rp in RETURN_PERIODS] for s in SCENARIOS}, index=RETURN_PERIODS)
ccnt = pd.DataFrame({s: [high_cnt(crit, rp, s) for rp in RETURN_PERIODS] for s in SCENARIOS}, index=RETURN_PERIODS)
print(f"Critical facilities in the High and Very High classes (%), n = {len(crit)}:"); display(cpct)

## 6. Road exposure

Percentages refer to the road surface area with an assigned hazard class.

In [ ]:
rrows = []
for rp in RETURN_PERIODS:
    for s in SCENARIOS:
        ra = road_area_by_class(cpath(rp, s)); tot = ra["ha"].sum()
        hi = ra[ra["Class"].isin(HIGH)]["ha"].sum()
        rrows.append({"RP": rp, "Scenario": s, "high_ha": round(hi, 3), "pct_high": round(100 * hi / tot, 2) if tot else 0})
roaddf = pd.DataFrame(rrows)
road_pct = roaddf.pivot(index="RP", columns="Scenario", values="pct_high").reindex(RETURN_PERIODS)[SCENARIOS]
road_ha = roaddf.pivot(index="RP", columns="Scenario", values="high_ha").reindex(RETURN_PERIODS)[SCENARIOS]
print("Road surface area in the High and Very High classes (%):"); display(road_pct)

## 7. Exposure-frequency curves

In [ ]:
xnum = [RP_YEARS[r] for r in RETURN_PERIODS]

def freq_curve(df, ylab, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for s in SCENARIOS: ax.plot(xnum, df[s].values, marker="o", label=s)
    ax.set_xscale("log"); ax.set_xticks(xnum); ax.set_xticklabels(RETURN_PERIODS)
    ax.set_xlabel("Return period"); ax.set_ylabel(ylab); ax.set_title(title); ax.grid(alpha=0.3)
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(OUT_DIR / fname, dpi=300); plt.show()

freq_curve(road_pct, "% road area in high hazard", "Road exposure-frequency curve", "fig_exposure_frequency_roads.png")
freq_curve(cpct, "% critical facilities in high hazard", "Critical-facility exposure-frequency curve",
           "fig_exposure_frequency_critical_facilities.png")
freq_curve(bpct, "% buildings in High+Very-High", "Building exposure-frequency curve",
           "fig_exposure_frequency_buildings.png")

## 8. Building exposure by land use

High-hazard share of the eight most frequent usage categories, excluding records without usage information, for the SSP5-8.5 (2051 to 2080) case.

In [ ]:
usages = (
    bld[bld[USAGE_FIELD] != "No Information"][USAGE_FIELD]
    .value_counts()
    .head(8)
    .index
    .tolist()
)
rows = []
for u in usages:
    sub = bld[bld[USAGE_FIELD] == u]
    rows.append([u] + [high_pct(sub, rp, FUTURE_CASE) for rp in RETURN_PERIODS])
usage_rp = pd.DataFrame(rows, columns=["Usage"] + RETURN_PERIODS).set_index("Usage")
display(usage_rp)

fig, ax = plt.subplots(figsize=(9, 5)); im = ax.imshow(usage_rp.values, cmap="OrRd", aspect="auto")
ax.set_xticks(range(len(RETURN_PERIODS))); ax.set_xticklabels(RETURN_PERIODS)
ax.set_yticks(range(len(usages))); ax.set_yticklabels(usages, fontsize=8)
for i in range(len(usages)):
    for j in range(len(RETURN_PERIODS)):
        ax.text(j, i, f"{usage_rp.values[i, j]:.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="% in high hazard"); ax.set_title(f"Usage exposure heatmap ({FUTURE_CASE})")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_usage_heatmap.png", dpi=300); plt.show()

## 9. High-hazard buildings by construction type, 100-year event

In [ ]:
struct = pd.DataFrame({s: bld[bld[f"100yr__{s}"].isin(HIGH)][STRUCT_FIELD].value_counts() for s in SCENARIOS}).fillna(0).astype(int)
total = struct.sum()
non_durable = struct.reindex(NON_DURABLE_TYPES).fillna(0).sum()
struct_table = struct.astype(object)
struct_table.loc["Total"] = total
struct_table.loc[" and ".join(NON_DURABLE_TYPES) + " (% of total)"] = (100 * non_durable / total).round(1)
struct_table

## 10. Frequency amplification

The target is the high-hazard exposure of the observed 100-year event.

In [ ]:
target_b = high_cnt(bld, "100yr", BASELINE)
target_c = high_cnt(crit, "100yr", BASELINE)
rows = []
for s in SCENARIOS:
    vb = {rp: high_cnt(bld, rp, s) for rp in RETURN_PERIODS}; Tb, ab = rp_shift(vb, target_b)
    vc = {rp: high_cnt(crit, rp, s) for rp in RETURN_PERIODS}; Tc, ac = rp_shift(vc, target_c)
    rows.append({"Scenario": s,
                 "Buildings: equivalent return period (yr)": round(Tb, 1), "Buildings: amplification factor": round(ab, 1),
                 "Critical facilities: equivalent return period (yr)": round(Tc, 1),
                 "Critical facilities: amplification factor": round(ac, 1)})
shift = pd.DataFrame(rows).set_index("Scenario")
print(f"Observed 100-year event: {target_b} buildings and {target_c} critical facilities in high hazard"); display(shift)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(len(SCENARIOS)), shift["Buildings: amplification factor"], color="#d7263d", alpha=0.85)
ax.set_xticks(range(len(SCENARIOS))); ax.set_xticklabels(SCENARIOS, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Frequency amplification factor"); ax.set_title("Frequency amplification of the observed 100-year exposure")
for i, v in enumerate(shift["Buildings: amplification factor"]): ax.text(i, v + 0.02, f"{v}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_frequency_amplification.png", dpi=300); plt.show()

## 11. Expected annual exposure

EAE is computed for buildings (buildings per year), critical facilities (facilities per year) and road surface (ha per year). The ratio of future to observed building EAE and their difference measure the change in risk.

In [ ]:
eae_rows = []
for s in SCENARIOS:
    eb = EAE({rp: high_cnt(bld, rp, s) for rp in RETURN_PERIODS})
    ec = EAE({rp: high_cnt(crit, rp, s) for rp in RETURN_PERIODS})
    er = EAE({rp: road_ha.loc[rp, s] for rp in RETURN_PERIODS})
    eae_rows.append({"Scenario": s, "EAE_buildings_per_yr": round(eb, 1), "EAE_criticals_per_yr": round(ec, 2),
                     "EAE_road_ha_per_yr": round(er, 2)})
eae = pd.DataFrame(eae_rows).set_index("Scenario")
base = eae.loc[BASELINE]
eae["Building EAE relative to observed"] = (eae["EAE_buildings_per_yr"] / base["EAE_buildings_per_yr"]).round(2)
eae["Increase in building EAE (buildings/yr)"] = (eae["EAE_buildings_per_yr"] - base["EAE_buildings_per_yr"]).round(1)
display(eae)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(range(len(SCENARIOS)), eae["EAE_buildings_per_yr"], color="#2c7fb8", alpha=0.85)
ax.set_xticks(range(len(SCENARIOS))); ax.set_xticklabels(SCENARIOS, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Buildings in high hazard / year"); ax.set_title("Expected Annual Exposure (EAE) of buildings")
for i, v in enumerate(eae["EAE_buildings_per_yr"]): ax.text(i, v + 0.3, f"{v}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_eae_buildings.png", dpi=300); plt.show()

## 12. Contribution of each probability interval to the building EAE

The trapezoidal sum is split into the intervals bounded by the anchor at $p = 1$ and the simulated events. The last column gives the share contributed by events of 10-year return period or more frequent.

In [ ]:
edges = ["p = 1"] + [f"{RP_YEARS[r]}-yr" for r in RETURN_PERIODS]
labels = [f"{a} to {b}" for a, b in zip(edges[:-1], edges[1:])]
n_frequent = sum(RP_YEARS[r] <= 10 for r in RETURN_PERIODS)       # intervals up to the 10-year event

contrib = pd.DataFrame({s: EAE_intervals({rp: high_cnt(bld, rp, s) for rp in RETURN_PERIODS}) for s in SCENARIOS},
                       index=labels).T
contrib.loc[f"Increase, {FUTURE_CASE} minus {BASELINE}"] = contrib.loc[FUTURE_CASE] - contrib.loc[BASELINE]
contrib["Total"] = contrib[labels].sum(axis=1)
contrib["Events of 10-yr or more frequent (%)"] = 100 * contrib[labels[:n_frequent]].sum(axis=1) / contrib["Total"]
eae_intervals = contrib.map(lambda v: round(float(v), 1))
eae_intervals

## 13. Building EAE by ward

In [ ]:
wards = sorted(bld[WARD_FIELD].dropna().unique())
wrows = []
for w in wards:
    sub = bld[bld[WARD_FIELD] == w]
    for s in SCENARIOS:
        e = EAE({rp: int(sub[f"{rp}__{s}"].isin(HIGH).sum()) for rp in RETURN_PERIODS})
        wrows.append({"Ward": int(w), "Scenario": s, "EAE": round(e, 1)})
wardeae = pd.DataFrame(wrows).pivot(index="Ward", columns="Scenario", values="EAE")[SCENARIOS]

ax = wardeae.plot(kind="bar", figsize=(9, 5), width=0.8)
ax.set_ylabel("EAE (buildings in high hazard / year)"); ax.set_title("Building EAE by ward")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_ward_eae.png", dpi=300); plt.show()

ward_table = wardeae.sort_values(BASELINE, ascending=False).copy()
ward_table[f"Change, {FUTURE_CASE} (buildings/yr)"] = (ward_table[FUTURE_CASE] - ward_table[BASELINE]).round(1)
ward_table[f"Change, {FUTURE_CASE} (%)"] = (100 * (ward_table[FUTURE_CASE] / ward_table[BASELINE] - 1)).round(1)
ward_table

## 14. Export

In [ ]:
with pd.ExcelWriter(OUT_DIR / "exposure_risk_results.xlsx", engine="openpyxl") as xw:
    bpct.to_excel(xw, sheet_name="building_pct")
    bcnt.to_excel(xw, sheet_name="building_count")
    bpct_inventory.to_excel(xw, sheet_name="building_pct_inventory")
    cpct.to_excel(xw, sheet_name="critical_pct")
    ccnt.to_excel(xw, sheet_name="critical_count")
    road_pct.to_excel(xw, sheet_name="road_pct")
    road_ha.to_excel(xw, sheet_name="road_high_ha")
    usage_rp.to_excel(xw, sheet_name="usage_by_return_period")
    struct_table.to_excel(xw, sheet_name="construction_type_100yr")
    shift.to_excel(xw, sheet_name="frequency_amplification")
    eae.to_excel(xw, sheet_name="EAE")
    eae_intervals.to_excel(xw, sheet_name="EAE_by_interval")
    ward_table.to_excel(xw, sheet_name="ward_EAE")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))